# Notebook 03 — Cumulative Abnormal Returns (CARs)

Now that the daily ARs have been computed in the previous notebook, they are aggregated into Cumulative Abnormal Returns (CARs) over different windows. The windows considered are:

| Window | Interpretation |
|--------|---------------|
| CAR[-20, -1] | Pre-event drift, so did the market already start reacting before the filing? |
| CAR[0, +1] | Immediate reaction (announcement effect) |
| CAR[0, +5] | Short-term reaction (one trading week) |
| CAR[0, +10] | Medium-term reaction (two weeks) |
| CAR[0, +20] | Longer-term post-event drift |
| CAR[-20, +20] | Full event window |

The "duplicate event" problem is also handled here, as some (ticker, date) pairs have both a Sale and a Sale+OE filing. Three different modes are created for dealing with this:
- **Both**: duplicates are kept in both groups (Sale and Sale+OE each get a copy)
- **Exclude**: conflicting events are dropped entirely
- **Merge**: conflicts are relabeled as "Sale+All"

Finally, the overlap filters from notebook 01 are applied to NaN out CARs that would be contaminated by nearby events.

**Inputs:**
- `event_study_abnormal_returns.csv`: daily ARs (traditional)
- `event_study_metadata.csv`: event metadata (alpha, beta, event type)
- `llm_abnormal_returns.csv`: daily ARs from the LLM-based approach
- `unique_event_filings.csv`: for overlap flag computation

**Outputs:**
- `cumulative_abnormal_returns_{mode}.csv`: unfiltered CARs for each duplicate-handling mode
- `cumulative_abnormal_returns_{mode}_clean.csv`: overlap-filtered CARs

In [1]:
from datetime import timedelta

import numpy as np
import pandas as pd

CAR_WINDOWS = {
    "CAR_pre20_pre1": (-20, -1),
    "CAR_0_1": (0, 1),
    "CAR_0_5": (0, 5),
    "CAR_0_10": (0, 10),
    "CAR_0_20": (0, 20),
    "CAR_pre20_20": (-20, 20),
}

In [2]:
def compute_cars(ar_group, event_date, ar_col="AR"):
    cars = {}
    for name, (start, end) in CAR_WINDOWS.items():
        car_start = event_date + timedelta(days=start)
        car_end = event_date + timedelta(days=end)
        mask = (ar_group["date"] >= car_start) & (ar_group["date"] <= car_end)
        window_data = ar_group.loc[mask]
        cars[name] = window_data[ar_col].sum() if len(window_data) > 0 else np.nan
    return cars


In [3]:
# traditional day-level ARs
trad_ar_df = pd.read_csv("event_study_abnormal_returns.csv", parse_dates=["event_date", "date"])
print(f"Traditional ARs: {len(trad_ar_df)} rows, {trad_ar_df.groupby(['ticker', 'event_date']).ngroups} events")

# traditional CARs for event_type + alpha/beta metadata
trad_car_meta = pd.read_csv("event_study_metadata.csv", parse_dates=["event_date"])
print(f"Traditional CARs metadata: {len(trad_car_meta)} events")

# LLM day-level ARs
llm_ar_df = pd.read_csv("llm_abnormal_returns.csv", parse_dates=["event_date", "date"])
print(f"LLM ARs: {len(llm_ar_df)} rows, {llm_ar_df.groupby(['ticker', 'event_date']).ngroups} events")
print(f"  Models: {llm_ar_df['model'].nunique()} — {list(llm_ar_df['model'].unique())}")
print(f"  Prompt variants: {llm_ar_df['prompt_variant'].nunique()} — {list(llm_ar_df['prompt_variant'].unique())}")

Traditional ARs: 19474 rows, 456 events
Traditional CARs metadata: 492 events
LLM ARs: 19474 rows, 456 events
  Models: 1 — ['mistral-small-3.2-24b']
  Prompt variants: 1 — ['with_formula']


## CARs — All Events (Unfiltered)

CARs are computed for both the traditional and LLM-based ARs by simply summing up the daily ARs within each window. At this stage, overlapping events are not filtered out yet — that comes later. First the raw numbers are generated.

In [4]:
# Deduplicate day-level ARs: the AR file can have duplicate rows when the same
# (ticker, event_date) has multiple trade types. drop duplicates so each
# (ticker, event_date, date) has exactly one AR value.
trad_ar_dedup = trad_ar_df.drop_duplicates(subset=["ticker", "event_date", "date"])

# Compute traditional CARs (one row per unique event)
trad_rows = []
for (ticker, event_date), group in trad_ar_dedup.groupby(["ticker", "event_date"]):
    cars = compute_cars(group, event_date, ar_col="AR")
    trad_rows.append({"ticker": ticker, "event_date": event_date, **cars})

trad_cars_base = pd.DataFrame(trad_rows)
print(f"Traditional CARs computed: {len(trad_cars_base)} unique (ticker, event_date) events")

Traditional CARs computed: 456 unique (ticker, event_date) events


In [5]:
# compute LLM CARs (one row per unique event per model/variant)
llm_rows = []
group_keys = ["ticker", "event_date", "model", "prompt_variant"]
for (ticker, event_date, model, prompt_variant), group in llm_ar_df.groupby(group_keys):
    llm_cars = compute_cars(group, event_date, ar_col="llm_AR")
    trad_cars = compute_cars(group, event_date, ar_col="traditional_AR")
    trad_cars_renamed = {f"traditional_{k}": v for k, v in trad_cars.items()}

    first = group.iloc[0]
    llm_rows.append({
        "ticker": ticker,
        "event_date": event_date,
        "model": model,
        "prompt_variant": prompt_variant,
        "llm_alpha": first["llm_alpha"],
        "llm_beta": first["llm_beta"],
        "traditional_alpha": first["traditional_alpha"],
        "traditional_beta": first["traditional_beta"],
        **llm_cars,
        **trad_cars_renamed,
    })

llm_cars_base = pd.DataFrame(llm_rows)
print(f"LLM CARs computed: {len(llm_cars_base)} rows ({llm_cars_base.groupby(['ticker', 'event_date']).ngroups} unique events)")

LLM CARs computed: 456 rows (456 unique events)


In [6]:
# --- Identify conflict events and build mode-aware views ---
# The metadata has 492 rows (36 (ticker, event_date) pairs have both Sale and Sale+OE).
# The base CARs have 456 rows (one per unique event). We need 3 modes:
#   - both:    keep conflict events in both groups (492 trad, 492 LLM)
#   - exclude: remove conflict events entirely (420 trad, 420 LLM)
#   - merge:   relabel Sale+OE as "Sale+All" for conflict events (456 trad, 456 LLM)

meta_cols = ["ticker", "event_date", "event_type", "alpha", "beta"]
event_meta = trad_car_meta[meta_cols].copy()

# Identify conflict (ticker, event_date) pairs
event_counts = event_meta.groupby(["ticker", "event_date"]).size()
conflict_pairs = set(event_counts[event_counts > 1].index)
print(f"Conflict (ticker, event_date) pairs: {len(conflict_pairs)}")

# --- BOTH mode: join metadata as-is (one-to-many for conflicts) ---
trad_both = trad_cars_base.merge(event_meta, on=["ticker", "event_date"], how="left")
trad_both = trad_both[["ticker", "event_date", "event_type", "alpha", "beta"] + list(CAR_WINDOWS.keys())]

# For LLM: also expand conflict events into 2 rows
llm_both = llm_cars_base.merge(
    event_meta[["ticker", "event_date", "event_type"]], on=["ticker", "event_date"], how="left"
)

# --- EXCLUDE mode: drop conflict events ---
trad_exclude = trad_both[
    ~trad_both.apply(lambda r: (r["ticker"], r["event_date"]) in conflict_pairs, axis=1)
].copy()
llm_exclude = llm_both[
    ~llm_both.apply(lambda r: (r["ticker"], r["event_date"]) in conflict_pairs, axis=1)
].copy()

# --- MERGE mode: relabel conflict events as "Sale+All", keep non-conflict as-is ---
# For non-conflict events, keep original type. For conflicts, both Sale and Sale+OE
# become "Sale+All". Since base has 1 row per event, just assign the merged label.
trad_merge = trad_cars_base.copy()
# Use a single event_type per event: for non-conflicts, take the unique type;
# for conflicts, assign "Sale+All"
event_type_unique = event_meta.drop_duplicates(subset=["ticker", "event_date"], keep="first")
trad_merge = trad_merge.merge(
    event_type_unique[["ticker", "event_date", "event_type", "alpha", "beta"]],
    on=["ticker", "event_date"], how="left"
)
trad_merge.loc[
    trad_merge.apply(lambda r: (r["ticker"], r["event_date"]) in conflict_pairs, axis=1),
    "event_type"
] = "Sale+All"
trad_merge = trad_merge[["ticker", "event_date", "event_type", "alpha", "beta"] + list(CAR_WINDOWS.keys())]

llm_merge = llm_cars_base.copy()
llm_merge = llm_merge.merge(
    event_type_unique[["ticker", "event_date", "event_type"]],
    on=["ticker", "event_date"], how="left"
)
llm_merge.loc[
    llm_merge.apply(lambda r: (r["ticker"], r["event_date"]) in conflict_pairs, axis=1),
    "event_type"
] = "Sale+All"

print(f"\nMode sample sizes (traditional):")
print(f"  Both:    {len(trad_both)} rows — {dict(trad_both['event_type'].value_counts())}")
print(f"  Exclude: {len(trad_exclude)} rows — {dict(trad_exclude['event_type'].value_counts())}")
print(f"  Merge:   {len(trad_merge)} rows — {dict(trad_merge['event_type'].value_counts())}")
print(f"\nMode sample sizes (LLM):")
print(f"  Both:    {len(llm_both)} rows — {dict(llm_both['event_type'].value_counts())}")
print(f"  Exclude: {len(llm_exclude)} rows — {dict(llm_exclude['event_type'].value_counts())}")
print(f"  Merge:   {len(llm_merge)} rows — {dict(llm_merge['event_type'].value_counts())}")

Conflict (ticker, event_date) pairs: 36

Mode sample sizes (traditional):
  Both:    492 rows — {'S - Sale': 315, 'S - Sale+OE': 165, 'P - Purchase': 12}
  Exclude: 420 rows — {'S - Sale': 279, 'S - Sale+OE': 129, 'P - Purchase': 12}
  Merge:   456 rows — {'S - Sale': 279, 'S - Sale+OE': 129, 'Sale+All': 36, 'P - Purchase': 12}

Mode sample sizes (LLM):
  Both:    492 rows — {'S - Sale': 315, 'S - Sale+OE': 165, 'P - Purchase': 12}
  Exclude: 420 rows — {'S - Sale': 279, 'S - Sale+OE': 129, 'P - Purchase': 12}
  Merge:   456 rows — {'S - Sale': 279, 'S - Sale+OE': 129, 'Sale+All': 36, 'P - Purchase': 12}


36 conflict pairs were identified — (ticker, event_date) combinations where both a Sale and a Sale+OE occurred. In "exclude" mode this brings the sample from 492 down to 420 events. In "merge" mode the conflicts are relabeled as "Sale+All", retaining all 456 unique (ticker, date) events. The LLM data has the same structure (one model, one prompt variant), so the row counts mirror the traditional side in each mode.

In [7]:
# descriptive stats by event type for each mode (Traditional CARs)
car_cols = list(CAR_WINDOWS.keys())

for mode_name, df in [("BOTH", trad_both), ("EXCLUDE", trad_exclude), ("MERGE", trad_merge)]:
    print(f"\n{'='*70}")
    print(f"Traditional CARs — {mode_name} mode ({len(df)} events)")
    for event_type, subset in df.groupby("event_type"):
        print(f"\n--- {event_type} (N={len(subset)}) ---")
        print(subset[car_cols].describe().loc[["count", "mean", "50%", "std", "min", "max"]].round(6))

print("LLM CARs — descriptive stats by event type and model")

for mode_name, df in [("BOTH", llm_both), ("EXCLUDE", llm_exclude), ("MERGE", llm_merge)]:
    print(f"\n--- {mode_name} mode ({len(df)} rows) ---")
    for (event_type, model), subset in df.groupby(["event_type", "model"]):
        print(f"\n  {event_type} / {model} (N={len(subset)})")
        print(subset[car_cols].describe().loc[["count", "mean", "50%", "std", "min", "max"]].round(6))


Traditional CARs — BOTH mode (492 events)

--- P - Purchase (N=12) ---
       CAR_pre20_pre1    CAR_0_1    CAR_0_5   CAR_0_10   CAR_0_20  \
count       12.000000  12.000000  12.000000  12.000000  12.000000   
mean        -0.018866   0.002988  -0.009039  -0.011280  -0.014509   
50%         -0.039094  -0.008545  -0.002498   0.002607  -0.024933   
std          0.065418   0.027765   0.037277   0.056656   0.050430   
min         -0.132952  -0.026336  -0.080478  -0.119116  -0.098425   
max          0.103631   0.064716   0.036910   0.063811   0.057520   

       CAR_pre20_20  
count     12.000000  
mean      -0.033375  
50%       -0.074868  
std        0.094736  
min       -0.143266  
max        0.148356  

--- S - Sale (N=315) ---
       CAR_pre20_pre1     CAR_0_1     CAR_0_5    CAR_0_10    CAR_0_20  \
count      315.000000  314.000000  315.000000  315.000000  315.000000   
mean         0.014599   -0.000077   -0.002406   -0.004400   -0.006940   
50%          0.010657   -0.000566   -0.001599

## Save Results 

The "unfiltered" CARs are saved for each mode.

In [8]:
# save unfiltered CARs for each mode
modes = {
    "both": (trad_both, llm_both),
    "exclude": (trad_exclude, llm_exclude),
    "merge": (trad_merge, llm_merge),
}

for mode_name, (trad_df, llm_df) in modes.items():
    trad_out = trad_df.copy()
    trad_out["source"] = "traditional"
    llm_out = llm_df.copy()
    llm_out["source"] = "llm"
    combined = pd.concat([trad_out, llm_out], ignore_index=True)
    fname = f"cumulative_abnormal_returns_{mode_name}.csv"
    combined.to_csv(fname, index=False)
    print(f"Saved {fname}: {len(combined)} rows (trad={len(trad_df)}, llm={len(llm_df)})")

Saved cumulative_abnormal_returns_both.csv: 984 rows (trad=492, llm=492)
Saved cumulative_abnormal_returns_exclude.csv: 840 rows (trad=420, llm=420)
Saved cumulative_abnormal_returns_merge.csv: 912 rows (trad=456, llm=456)


## Out-of-scope: Overlap-Filtered CARs

The overlap flags are applied to set contaminated windows to NaN. The idea is that if two events for the same stock are too close together, the CAR for the overlapping window isn't trustworthy, so it's marked as missing rather than using a potentially biased number. The "clean" files are what gets used for the actual statistical tests.

In [9]:
def compute_overlap_flags(events_df):
    df = events_df.copy()
    df = df.sort_values(["Ticker", "Trade Type", "Filing Date"]).reset_index(drop=True)

    df["days_since_prev"] = df.groupby(["Ticker", "Trade Type"])["Filing Date"].diff().dt.days
    df["days_until_next"] = (
        df.groupby(["Ticker", "Trade Type"])["Filing Date"].diff(-1).dt.days.abs()
    )

    def has_overlap_forward(row, threshold):
        return pd.notna(row["days_until_next"]) and row["days_until_next"] < threshold

    def has_overlap_backward(row, threshold):
        return pd.notna(row["days_since_prev"]) and row["days_since_prev"] < threshold

    def has_overlap_bidirectional(row, threshold):
        return has_overlap_forward(row, threshold) or has_overlap_backward(row, threshold)

    df["overlap_0_1"] = df.apply(lambda row: has_overlap_forward(row, 2), axis=1)
    df["overlap_0_5"] = df.apply(lambda row: has_overlap_forward(row, 6), axis=1)
    df["overlap_0_10"] = df.apply(lambda row: has_overlap_forward(row, 11), axis=1)
    df["overlap_0_20"] = df.apply(lambda row: has_overlap_forward(row, 21), axis=1)
    df["overlap_pre20_pre1"] = df.apply(lambda row: has_overlap_backward(row, 21), axis=1)
    df["overlap_pre20_20"] = df.apply(lambda row: has_overlap_bidirectional(row, 21), axis=1)

    return df

In [10]:
events_df = pd.read_csv("unique_event_filings.csv", sep=",")
events_df["Filing Date"] = pd.to_datetime(events_df["Filing Date"], format="%d.%m.%Y")

events_with_flags = compute_overlap_flags(events_df)
events_with_flags["event_date"] = events_with_flags["Filing Date"]
events_with_flags["ticker"] = events_with_flags["Ticker"]

OVERLAP_TO_CAR = {
    "overlap_pre20_pre1": "CAR_pre20_pre1",
    "overlap_0_1": "CAR_0_1",
    "overlap_0_5": "CAR_0_5",
    "overlap_0_10": "CAR_0_10",
    "overlap_0_20": "CAR_0_20",
    "overlap_pre20_20": "CAR_pre20_20",
}

print("Overlap summary:")
print(f"{'Window':<20} {'Overlapping':>12} {'Clean':>8} {'Total':>8} {'% Overlap':>10}")
for overlap_col, car_name in OVERLAP_TO_CAR.items():
    n_overlap = events_with_flags[overlap_col].sum()
    n_total = len(events_with_flags)
    n_clean = n_total - n_overlap
    print(f"{car_name:<20} {n_overlap:>12} {n_clean:>8} {n_total:>8} {n_overlap/n_total*100:>9.1f}%")

# build overlapping sets per window
overlap_sets = {}
for overlap_col, car_name in OVERLAP_TO_CAR.items():
    overlapping = events_with_flags.loc[
        events_with_flags[overlap_col], ["ticker", "event_date"]
    ]
    overlap_sets[car_name] = set(zip(overlapping["ticker"], overlapping["event_date"]))

def apply_overlap_filter(df, car_cols_to_filter):
    out = df.copy()
    for car_name in car_cols_to_filter:
        if car_name not in overlap_sets:
            continue
        mask = out.apply(
            lambda row: (row["ticker"], row["event_date"]) in overlap_sets[car_name], axis=1
        )
        out.loc[mask, car_name] = np.nan
        # Also NaN the traditional_ prefixed columns for LLM DataFrames
        trad_col = f"traditional_{car_name}"
        if trad_col in out.columns:
            out.loc[mask, trad_col] = np.nan
    return out

# apply overlap filtering to each mode
clean_modes = {}
for mode_name, (trad_df, llm_df) in modes.items():
    clean_trad = apply_overlap_filter(trad_df, list(CAR_WINDOWS.keys()))
    clean_llm = apply_overlap_filter(llm_df, list(CAR_WINDOWS.keys()))
    clean_modes[mode_name] = (clean_trad, clean_llm)


print("\nComparison: All vs Clean (Traditional, EXCLUDE mode)")
trad_ex, _ = modes["exclude"]
clean_trad_ex, _ = clean_modes["exclude"]
print(f"{'Window':<20} {'All Mean':>12} {'Clean Mean':>12} {'All N':>8} {'Clean N':>8}")
print("-" * 62)
for car_name in CAR_WINDOWS:
    all_mean = trad_ex[car_name].mean()
    clean_mean = clean_trad_ex[car_name].mean()
    all_n = trad_ex[car_name].notna().sum()
    clean_n = clean_trad_ex[car_name].notna().sum()
    print(f"{car_name:<20} {all_mean:>12.6f} {clean_mean:>12.6f} {all_n:>8} {clean_n:>8}")

Overlap summary:
Window                Overlapping    Clean    Total  % Overlap
CAR_pre20_pre1                332      160      492      67.5%
CAR_0_1                        58      434      492      11.8%
CAR_0_5                       171      321      492      34.8%
CAR_0_10                      272      220      492      55.3%
CAR_0_20                      332      160      492      67.5%
CAR_pre20_20                  403       89      492      81.9%

Comparison: All vs Clean (Traditional, EXCLUDE mode)
Window                   All Mean   Clean Mean    All N  Clean N
--------------------------------------------------------------
CAR_pre20_pre1           0.014514     0.015354      420      136
CAR_0_1                  0.000795     0.000732      419      373
CAR_0_5                  0.000675    -0.001938      420      283
CAR_0_10                 0.000098    -0.006485      420      198
CAR_0_20                -0.000465    -0.003260      420      144
CAR_pre20_20             0.014049  

In [11]:
# save clean (overlap-filtered) results for each mode
for mode_name, (clean_trad, clean_llm) in clean_modes.items():
    trad_out = clean_trad.copy()
    trad_out["source"] = "traditional"
    llm_out = clean_llm.copy()
    llm_out["source"] = "llm"
    combined = pd.concat([trad_out, llm_out], ignore_index=True)
    fname = f"cumulative_abnormal_returns_{mode_name}_clean.csv"
    combined.to_csv(fname, index=False)
    print(f"Saved {fname}: {len(combined)} rows")

# events excluded per window (using exclude mode as example)
clean_trad_ex, _ = clean_modes["exclude"]
trad_ex, _ = modes["exclude"]
print("\nEvents excluded by overlap filter (EXCLUDE mode, traditional):")
for car_name in CAR_WINDOWS:
    all_n = trad_ex[car_name].notna().sum()
    clean_n = clean_trad_ex[car_name].notna().sum()
    excluded = all_n - clean_n
    print(f"  {car_name}: {excluded} excluded ({excluded/all_n*100:.1f}%)")

Saved cumulative_abnormal_returns_both_clean.csv: 984 rows
Saved cumulative_abnormal_returns_exclude_clean.csv: 840 rows
Saved cumulative_abnormal_returns_merge_clean.csv: 912 rows

Events excluded by overlap filter (EXCLUDE mode, traditional):
  CAR_pre20_pre1: 284 excluded (67.6%)
  CAR_0_1: 46 excluded (11.0%)
  CAR_0_5: 137 excluded (32.6%)
  CAR_0_10: 222 excluded (52.9%)
  CAR_0_20: 276 excluded (65.7%)
  CAR_pre20_20: 340 excluded (81.0%)


The overlap analysis reveals a substantial data-quality tradeoff. For the short CAR[0,1] window only about 11% of events are flagged as overlapping, leaving most of the sample intact. But for wider windows the contamination rate climbs steeply: roughly 55% for CAR[0,10] and over 80% for the full CAR[-20,+20] window. After overlap filtering in "exclude" mode, the widest window retains only 80 clean events, which is a fairly thin sample.